
# XGBoost Hyperparameter Tuning

This notebook performs hyperparameter tuning for the match-level XGBoost classifier and stores the tuned model along with updated metadata.


In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from xgboost import XGBClassifier


In [ ]:
# Load and inspect the processed match-level features.
raw_df = pd.read_csv("test.csv")
raw_df.head()


In [ ]:
def prepare_features(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, List[str]]:
    df = df.copy()

    # Drop identifiers that are not predictive
    drop_cols = [col for col in ["Unnamed: 0", "match_id"] if col in df.columns]
    df = df.drop(columns=drop_cols)

    # Separate target
    y = df.pop("y_match").astype(int)

    # Ensure boolean columns are represented as integers
    bool_cols = df.select_dtypes(include="bool").columns
    if len(bool_cols):
        df[bool_cols] = df[bool_cols].astype(int)

    # Convert remaining categorical columns to numeric codes
    cat_cols = df.select_dtypes(include="object").columns
    for col in cat_cols:
        df[col] = df[col].astype("category").cat.codes

    feature_columns = df.columns.tolist()
    return df, y, feature_columns

X, y, feature_columns = prepare_features(raw_df)
X.head()


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_valid.shape


In [ ]:
param_distributions: Dict[str, List] = {
    "model__max_depth": [3, 4, 5, 6, 8, 10],
    "model__learning_rate": np.linspace(0.01, 0.3, 10),
    "model__n_estimators": [200, 400, 600, 800, 1000],
    "model__subsample": np.linspace(0.6, 1.0, 5),
    "model__colsample_bytree": np.linspace(0.6, 1.0, 5),
    "model__min_child_weight": [1, 3, 5, 7],
    "model__gamma": [0, 0.25, 0.5, 1.0],
}

base_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    enable_categorical=False,
    n_jobs=-1,
)

pipeline = Pipeline([
    ("identity", FunctionTransformer(lambda df: df, validate=False)),
    ("model", base_model),
])

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="roc_auc",
    n_jobs=-1,
    cv=3,
    verbose=1,
    random_state=42,
)
search.fit(X_train, y_train)
search.best_params_


In [ ]:
best_model: Pipeline = search.best_estimator_

valid_pred_proba = best_model.predict_proba(X_valid)[:, 1]
valid_pred = (valid_pred_proba >= 0.5).astype(int)

metrics = {
    "log_loss": log_loss(y_valid, valid_pred_proba),
    "roc_auc": roc_auc_score(y_valid, valid_pred_proba),
    "accuracy": accuracy_score(y_valid, valid_pred),
}
metrics


In [ ]:
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

model_filename = f"xgb_tuned_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
model_path = models_dir / model_filename

# XGBoost save_model requires the underlying booster.
best_model.named_steps["model"].save_model(model_path)
model_path


In [ ]:
metadata_path = Path("model_data.parquet")
model_df = pd.read_parquet(metadata_path, engine="pyarrow") if metadata_path.exists() else pd.DataFrame()

next_model_num = (model_df["model_num"].max()  1) if not model_df.empty else 1
new_row = pd.DataFrame(
    {
        "model_num": [int(next_model_num)],
        "model_desc": ["Tuned XGBoost classifier"],
        "model_path": [str(model_path)],
        "input_columns": ["$".join(feature_columns)],
    }
)

updated_df = pd.concat([model_df, new_row], ignore_index=True)
updated_df.to_parquet(metadata_path, engine="pyarrow", index=False)
updated_df.tail()
